# Infolingo

### Setup

In [262]:
%load_ext autoreload
%autoreload 2

In [263]:
from utils import *

[nltk_data] Downloading package punkt_tab to /Users/alice/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package brown to /Users/alice/nltk_data...
[nltk_data]   Package brown is already up-to-date!


In [232]:
import re
import nltk
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
nltk.download("punkt_tab")

# Evaluation Dataset
from datasets import load_dataset
ds = load_dataset("rajpurkar/squad_v2")

# Use Roberta Base Squad
from transformers import pipeline
pipe = pipeline("question-answering", model="deepset/roberta-base-squad2")

[nltk_data] Downloading package punkt_tab to /Users/alice/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


In [469]:
import argparse
import collections
import json
import numpy as np
import os
import string
import sys
from tqdm import tqdm
import random
from collections import Counter
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
random.seed(2024)

# Squad eval: https://storageclwsprod1.blob.core.windows.net/bundles/0x6b567e1cf2e041ec80d7098f031c5c9e/contents.gz?se=2024-11-28T00%3A36%3A38Z&sp=rt&sv=2019-12-12&sr=b&rscd=inline%3B%20filename%3D%22evaluate-v2.0.py%22&rsce=gzip&rsct=text/x-python&sig=6kzhUiIKQWEOZJ6C0VFyDlN/xFEpOKJsgq8GYCLzvfs%3D

def normalize_answer(s):
    """Lower text and remove punctuation, articles and extra whitespace."""

    def remove_articles(text):
        regex = re.compile(r"\b(a|an|the)\b", re.UNICODE)
        return re.sub(regex, " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    # Additional normalization
    s = s.replace(" _ ", " ")
    s = white_space_fix(remove_articles(remove_punc(lower(s))))
    s = [word for word in word_tokenize(s) if word not in stop_words]
    s = " ".join(s)

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def get_tokens(s):
    if not s:
        return []
    return normalize_answer(s).split()


def compute_exact(a_gold, a_pred):
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))


def compute_f1(a_gold, a_pred):
    gold_toks = get_tokens(a_gold)
    pred_toks = get_tokens(a_pred)
    common = collections.Counter(gold_toks) & collections.Counter(pred_toks)
    num_same = sum(common.values())
    if len(gold_toks) == 0 or len(pred_toks) == 0:
        # If either is no-answer, then F1 is 1 if they agree, 0 otherwise
        return int(gold_toks == pred_toks)
    if num_same == 0:
        return 0
    precision = 1.0 * num_same / len(pred_toks)
    recall = 1.0 * num_same / len(gold_toks)
    f1 = (2 * precision * recall) / (precision + recall)
    return f1


def make_eval_dict(exact_scores, f1_scores, qid_list=None):
    if not qid_list:
        total = len(exact_scores)
        return collections.OrderedDict(
            [
                ("exact", 100.0 * sum(exact_scores.values()) / total),
                ("f1", 100.0 * sum(f1_scores.values()) / total),
                ("total", total),
            ]
        )
    else:
        total = len(qid_list)
        return collections.OrderedDict(
            [
                ("exact", 100.0 * sum(exact_scores[k] for k in qid_list) / total),
                ("f1", 100.0 * sum(f1_scores[k] for k in qid_list) / total),
                ("total", total),
            ]
        )


def get_raw_scores(dataset, preds):
    exact_scores = {}
    f1_scores = {}
    for article in dataset:
        for p in article["paragraphs"]:
            for qa in p["qas"]:
                qid = qa["id"]
                gold_answers = [
                    a["text"] for a in qa["answers"] if normalize_answer(a["text"])
                ]
                if not gold_answers:
                    # For unanswerable questions, only correct answer is empty string
                    gold_answers = [""]
                if qid not in preds:
                    print("Missing prediction for %s" % qid)
                    continue
                a_pred = preds[qid]
                # Take max over all gold answers
                exact_scores[qid] = max(compute_exact(a, a_pred) for a in gold_answers)
                f1_scores[qid] = max(compute_f1(a, a_pred) for a in gold_answers)
    return exact_scores, f1_scores

In [520]:
def get_gold_answers(dataset, n=None, empty=False):
    """Creates an ID-to-gold answer map. Adapted from official eval code."""
    n = len(dataset) if n is None else min(len(dataset), n)
    gold_answers_map = {}
    i = -1
    while len(gold_answers_map) < n:
        i += 1
        qa = dataset[i]
        qid = qa["id"]
        gold_answers = [
            normalize_answer(a) for a in qa["answers"]["text"] if normalize_answer(a)
        ]
        if not gold_answers:
            if empty == False:
                continue
            # For unanswerable questions, only correct answer is empty string
            gold_answers = [""]
        gold_answers_map[qid] = gold_answers
    return gold_answers_map


def mask_text(text, vocab):
    """
    Masks words in a sentence not found in the provided vocabulary,
    preserving punctuation and numbers, and matching stems.

    Args:
        sentence (str): The input sentence.
        vocab (list): A list of words forming the vocabulary.
    Returns:
        str: The masked sentence.
    """
    # Initialize the stemmer and stem the vocabulary
    stemmer = PorterStemmer()
    stemmed_vocab = {stemmer.stem(word.lower()) for word in vocab}

    # Tokenize the sentence into words, punctuation, and numbers
    #tokens = re.findall(r"\b\w+\b|[^\w\s]", text)
    tokens = word_tokenize(text)

    # Mask tokens not in the stemmed vocabulary or not numbers
    masked_tokens = [
        (
            token
            if re.match(r"[^\w\s]", token) or token.isdigit()
            # or stemmer.stem(token.lower()) in stemmed_vocab # use stemmer when choosing word
            or token.lower() in vocab
            else "_"
        )
        for token in tokens
    ]

    # Reassemble the sentence while preserving punctuation
    return "".join(
        token if re.match(r"[^\w\s]", token) else f" {token}" for token in masked_tokens
    ).strip()


def create_qa_item(qa, vocab_func, target_pct, prior_vocab):
    item = {
        "question": qa["question"],
        "context": qa["context"],
    }
    vocab = get_vocab(vocab_func, item["context"], target_pct, prior_vocab)
    item["context"] = mask_text(item["context"], vocab=vocab+prior_vocab)
    return item, vocab


def tokenize_text(text):
    tokens = word_tokenize(text.lower())
    words = [word for word in tokens if word.isalnum()]
    words = [word for word in words if not word.isdigit()]
    return words

def get_word_stats(words):
    word_counts = Counter(words)
    total_words = sum(word_counts.values())
    word_probs = {word: count / total_words for word, count in word_counts.items()}
    return word_counts, word_probs

def take_test(dataset, vocab_func, target_pct, prior_vocab=[]):
    preds = {}
    vocabs = {}
    for i in tqdm(range(len(dataset))):
        qa = dataset[i]
        qid = qa["id"]

        item, vocab = create_qa_item(qa, vocab_func, target_pct, prior_vocab)
        a_pred = pipe(item)
        a_pred = a_pred["answer"]
        preds[qid] = a_pred
        vocabs[qid] = vocab
    return preds, vocabs


def get_scores(gold_answers_map, pred_answers_map):
    exact_scores = {}
    f1_scores = {}
    for qid in gold_answers_map:
        if qid not in pred_answers_map:
            continue

        gold_answers = gold_answers_map[qid]
        a_pred = pred_answers_map[qid]

        exact_scores[qid] = max(compute_exact(a, a_pred) for a in gold_answers)
        f1_scores[qid] = max(compute_f1(a, a_pred) for a in gold_answers)
    return exact_scores, f1_scores

### Vocab Selectors

In [528]:
def get_vocab(pick_next_func, text, target_pct, prior_vocab=[]):
    words = tokenize_text(text)
    words = [word for word in words if word not in prior_vocab] # prior knowledge

    word_counts, word_probs = get_word_stats(words)
    num_unique = len(word_counts)
    num_to_pick = int(target_pct * num_unique)

    vocab = pick_next_func(words, word_counts, word_probs, n=num_to_pick)
    return vocab


def pick_baseline_random(words, word_counts, word_probs, n=1):
    """Picks random next word."""
    picks = random.sample(list(word_counts.keys()), n)
    return picks


def pick_baseline_frequent(words, word_counts, word_probs, n=1):
    """Picks most frequent next word."""
    picks = word_counts.most_common(n)
    picks = [p[0] for p in picks]
    return picks

corpus_word_probs = get_corpus_word_probs()
def pick_entropy(words, word_counts, word_probs, n=1):
    def _compute_entropy(w_counts, w_probs):
        entropy = 0
        for w in w_counts:
            entropy += math.log2(1 / corpus_word_probs.get(w, 0.00001)) * w_probs[w]
        return entropy
    # baseline entropy of text
    baseline_entropy = _compute_entropy(word_counts, word_probs)
    deltas = []

    # remove each word in turn
    for word in word_counts:
        candidate_words = [w for w in words if w != word]
        candidate_word_counts, candidate_probs = get_word_stats(candidate_words)
        candidate_entropy = _compute_entropy(candidate_word_counts, candidate_probs)
        # print(f"H(X) after removing {word} is {candidate_entropy:.2f}")

        delta = baseline_entropy - candidate_entropy
        deltas.append((word, delta))

    deltas = sorted(deltas, key=lambda x: x[1], reverse=True)
    picks = [w[0] for w in deltas[:n]]
    return picks


def pick_kl_divergence(words, word_counts, word_probs, n=1):
    all_words = set(corpus_word_probs.keys()).union(set(words))

    smoothing = 0.00001
    P_smoothed = {w: corpus_word_probs.get(w, 0) + smoothing for w in all_words}
    Q_smoothed = {w: word_probs.get(w, 0) + smoothing for w in all_words}

    P_total = sum(P_smoothed.values())
    Q_total = sum(Q_smoothed.values())

    P_normalized = {w: p / P_total for w, p in P_smoothed.items()}
    Q_normalized = {w: q / Q_total for w, q in Q_smoothed.items()}

    kl_divergence = 0.0
    contributions = {}
    for w in all_words:
        q = Q_normalized[w]
        p = P_normalized[w]
        if q > 0:  # Skip if Q(w) is 0 (log(0) is undefined)
            contribution = q * np.log(q / p)
            kl_divergence += contribution
            contributions[w] = contribution

    sorted_words = sorted(contributions.items(), key=lambda x: x[1], reverse=True)
    picks = [w[0] for w in sorted_words[:n]]
    return picks

In [485]:
def load_vocab(filename):
    with open(filename, "r") as f:
        contents = f.read()
    vocab = contents.split("\n")
    return vocab


def save_result(contents, output_name):
    with open(output_name, "w") as f:
        file_contents = "\n".join(contents)
        f.write(file_contents)
        print(f"[+] Experiment stats saved to {output_name}")


def save_json(data, output_name):
    with open(output_name, "w") as json_file:
        json.dump(data, json_file, indent=4)
    print(f"[+] Experiment preds saved to {output_name}")

def load_json(file_path):
    try:
        with open(file_path, "r") as file:
            return json.load(file)
    except:
        return {}

## Experiment

In [ ]:
import time

# Setup
timestamp = str(int(time.time()))
output_csv = f"results/experiment_{timestamp}.csv"
output_json = f"results/experiment_{timestamp}.json"

n = 1000
dataset = ds["validation"]
gold_answers_map = get_gold_answers(dataset, n, empty=False)
qids = list(gold_answers_map.keys())
relevant_dataset = [item for item in dataset if item["id"] in qids]

print("Num gold answers:", len(gold_answers_map))
assert set(gold_answers_map.keys()) == set([item["id"] for item in relevant_dataset])

prior_vocab_name = "A1A2"
prior_vocab = load_vocab(f"vocab/{prior_vocab_name}.txt")
# prior_vocab_name = "none"
# prior_vocab = []

contents = ["n,prior_vocab,vocab_func,target_pct,accuracy,f1,avg_vocab_len"]  # header
json_predictions = {}

[nltk_data] Downloading package punkt_tab to /Users/alice/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package brown to /Users/alice/nltk_data...
[nltk_data]   Package brown is already up-to-date!


Num gold answers: 1000


In [535]:
# Run through baseline
vocab_func = pick_kl_divergence
vocab_name = vocab_func.__name__
json_predictions = load_json(output_json)
json_predictions[vocab_name] = {}
print(f"[*] Running experiments for {vocab_name}...")
#for target_pct in [0, 0.10, 0.25, 0.5, 0.75, 1.0]: # first run include none and all
for target_pct in [0.10, 0.25, 0.5, 0.75]:
    json_predictions[vocab_name][f"{target_pct}"] = {}
    pred_answers_map, vocabs = take_test(
        relevant_dataset,
        vocab_func=vocab_func,
        target_pct=target_pct,
        prior_vocab=prior_vocab
    )
    json_predictions[vocab_name][f"{target_pct}"]["predictions"] = pred_answers_map
    #json_predictions[vocab_name][f"{target_pct}"]["vocabs"] = vocabs

    exact_scores, f1_scores = get_scores(gold_answers_map, pred_answers_map)
    eval_dict = make_eval_dict(exact_scores, f1_scores)

    lens = [len(s) for s in vocabs.values()]
    avg_len = sum(lens) / len(lens)
    content = f"{n},{prior_vocab_name},{vocab_name},{target_pct},{eval_dict['exact']:.2f},{eval_dict['f1']:.2f},{avg_len:.2f}"
    print(f"\t{content}")
    contents.append(content)
    save_result(contents, output_csv)
    save_json(json_predictions, output_json)

[*] Running experiments for pick_kl_divergence...


  0%|          | 0/1000 [00:00<?, ?it/s]/Users/alice/Documents/05-STANFORD/00-Courses/2 SOPHMORE/05-SOPH-AUT/CS109/Challenge/InfoLingo/.env/lib/python3.9/site-packages/transformers/pipelines/question_answering.py:391: FutureWarning: Passing a list of SQuAD examples to the pipeline is deprecated and will be removed in v5. Inputs should be passed using the `question` and `context` keyword arguments instead.
  warnings.warn(
100%|██████████| 1000/1000 [04:59<00:00,  3.34it/s]


	1000,A1A2,pick_kl_divergence,0.1,22.10,36.32,3.53
[+] Experiment stats saved to results/experiment_1732947764.csv
[+] Experiment preds saved to results/experiment_1732947764.json


100%|██████████| 1000/1000 [05:05<00:00,  3.27it/s]


	1000,A1A2,pick_kl_divergence,0.25,35.60,50.10,9.58
[+] Experiment stats saved to results/experiment_1732947764.csv
[+] Experiment preds saved to results/experiment_1732947764.json


100%|██████████| 1000/1000 [05:28<00:00,  3.05it/s]


	1000,A1A2,pick_kl_divergence,0.5,49.80,65.09,19.63
[+] Experiment stats saved to results/experiment_1732947764.csv
[+] Experiment preds saved to results/experiment_1732947764.json


100%|██████████| 1000/1000 [05:02<00:00,  3.30it/s]


	1000,A1A2,pick_kl_divergence,0.75,63.40,76.38,29.45
[+] Experiment stats saved to results/experiment_1732947764.csv
[+] Experiment preds saved to results/experiment_1732947764.json
